# 01 · Authentic Bengali corpus, quality review, and exact token budget

Build a corpus of **up to 2,000,000 tokens** from Tagore's original Bengali writing.
This notebook orchestrates the actual implementation in `src/tagore_llm/collection.py`
and `corpus.py`. Raw downloads and authorized imports remain intact. No artificial
Q&A, paraphrases, spelling modernization, or invented literature enter the corpus.

**Repository-only defaults:** all network and preparation actions are off. Read the
README, install the environment, then explicitly enable the operations you want.
Running the default notebook does not download anything or start training.


In [ ]:
from pathlib import Path
import sys, json, logging

here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / "src/tagore_llm").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/ directory.")
sys.path.insert(0, str(ROOT / "src"))
from tagore_llm.common import load_config, read_json, read_jsonl, write_json
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
PROJECT = load_config(ROOT)
print("Project:", ROOT)
print("Pinned model:", PROJECT["model_id"], "revision:", PROJECT["revision"])


## Configuration and source policy

Wikisource's public-domain notices and transcription license are recorded separately.
Its current robots rules disallow `/w/` and `/api/` for this collector, so the default
implementation checks robots live and uses permitted article pages, serially, with a
minimum three-second delay. It stops on policy/redirect uncertainty. Author catalogs
provide discovery evidence; individual texts still need authorship, rights and QC checks.

SNLTR's FAQ permits individual-page downloads; its live footer also asserts rights.
Bichitra/Jadavpur is an authoritative edition reference, but no bulk reuse permission
was established. These sources use the authorized manual-import workflow instead
of an unverified crawler. See `docs/SOURCES_AND_COMPATIBILITY.md` for primary sources.


In [ ]:
RUN_COLLECTION = False
RUN_MANUAL_IMPORT = False
RUN_PREPARATION = False
# Set to an actual authorized manifest, never the unfilled example.
MANUAL_MANIFEST = ROOT / "configs/manual_import.local.jsonl"

# Edit configs/project.json for persistent settings. This copy can be changed for this session.
# PROJECT["collection"]["contact"] = "your actual contact email or project URL"
assert 1 <= PROJECT["max_tokens"] <= 2_000_000
assert PROJECT["sequence_length"] >= 8
print(json.dumps(PROJECT, indent=2, ensure_ascii=False))


## Resumable collection and authorized imports

Collection saves each response and SHA-256 checksum before extracting text. Repeated
runs reuse verified responses and document IDs; they do not append duplicate copies.
A real contact value is required in the user agent. 429/503 responses trigger backoff;
authentication, paywalls, denied paths and anti-bot controls are never bypassed.

For manual imports, fill every metadata field, set `authorized=true` and
`reviewed_original_text=true` only after checking the actual material and permission.
The example's false flags deliberately prevent accidental import.


In [ ]:
if RUN_COLLECTION:
    from tagore_llm.collection import collect
    print(collect(ROOT, PROJECT))
else:
    print("Collection disabled; no network requests.")

if RUN_MANUAL_IMPORT:
    from tagore_llm.collection import import_manual
    print("Imported document IDs:", import_manual(ROOT, MANUAL_MANIFEST))
else:
    print("Manual import disabled.")


## Quality control and the manual review queue

Cleaning preserves Bengali characters, historical wording, joiners, punctuation,
paragraph boundaries, line breaks and stanza boundaries. It does not apply NFC/NFKC.
Only the initial BOM, newline conventions, HTML NBSP, trailing layout whitespace and
excessive blank lines are normalized; each change is logged. Known site UI is removed
structurally. Suspected OCR, corrupted text, tables, front matter and uncertain rights
enter review, rather than speculative correction. Containers/TOCs are excluded.

Exact and near duplicate editions are removed. Remaining works that share a contiguous
16-word passage are connected before splitting, preserving legitimate refrains/quotations.
SQLite indexes keep duplicate fingerprints off the GPU and out of a giant Python map.

A paragraph or stanza that exceeds the configured sequence length is queued for review.
The pipeline does not truncate it. Import verified natural sections if needed, with the
**same canonical work_id** for all sections and editions. Do not approve text simply
to fill the target budget. Entire selected documents are retained or excluded by budget.


In [ ]:
if RUN_PREPARATION:
    from tagore_llm.common import tokenizer_for
    from tagore_llm.corpus import prepare
    # This explicit action downloads the pinned TOKENIZER if it is not already cached.
    tokenizer = tokenizer_for(PROJECT)
    preparation = prepare(ROOT, PROJECT, tokenizer)
    print(json.dumps(preparation["quality"], indent=2, ensure_ascii=False))
    print(json.dumps(preparation["statistics"], indent=2, ensure_ascii=False))
else:
    print("Preparation disabled; tokenizer is not downloaded.")


In [ ]:
queue_path = ROOT / "data/metadata/manual_review_queue.jsonl"
rejected_path = ROOT / "data/processed/rejected_documents.jsonl"
if queue_path.exists():
    review_rows = list(read_jsonl(queue_path))
    print("Review queue:", len(review_rows))
    for row in review_rows[:10]:
        print(row["doc_id"], row.get("title"), row["reasons"], row["content_checksum"])
else:
    print("No review queue yet.")
if rejected_path.exists():
    from collections import Counter
    print("Rejection reasons:", Counter(reason for row in read_jsonl(rejected_path) for reason in row.get("reasons", [])))


To review a flagged work, compare `data/cleaned/<doc_id>.json` with the original source
and its saved HTML. Put one decision per document in `data/metadata/review_decisions.jsonl`:

```json
{"doc_id":"actual ID", "decision":"approve", "content_checksum":"actual cleaned checksum", "reviewer":"your name", "reviewed_original_text":true, "rights_evidence_url":"actual permission or license URL"}
```

Use `decision="reject"` for exclusions. Corrupted Unicode or mojibake cannot be approved
away: obtain a corrected authorized source and import it with provenance. Re-run
preparation after review. Never edit frozen processed files to make an audit pass.


## Budget and leakage audit

The sequence constructor tokenizes each actual training passage with the pinned Qwen
tokenizer, appends one EOS, and counts that exact array. No BOS, chat template or prompt
wrapper is added. Padding positions are masked later and are reported separately from
literary data. Independent work groups target a 90/10 token split; large works may
make the achieved fraction differ. Fewer authentic tokens are a valid result.


In [ ]:
if RUN_PREPARATION:
    from tagore_llm.corpus import validate_dataset
    audit = validate_dataset(ROOT, PROJECT, tokenizer)
    print(json.dumps(audit, indent=2))
    assert audit["total_tokens"] <= 2_000_000
stats_path = ROOT / "data/processed/token_statistics.json"
if stats_path.exists():
    print(json.dumps(read_json(stats_path), indent=2, ensure_ascii=False))
else:
    print("No dataset has been prepared. No corpus or statistics are fabricated.")


## Output contract

A successful preparation produces `train.jsonl`, `validation.jsonl`, `metadata.jsonl`,
`dataset_manifest.json`, `quality_report.json`, `rejected_documents.jsonl`,
`token_statistics.json`, and `corpus.txt` under `data/processed/`. Review queues,
deduplication groups, source records and budget exclusions remain under `data/metadata/`.
No URLs, licenses, IDs or technical metadata are inserted into the `text` passages.
Continue to notebook 02 only after auditing these artifacts.
